In [1]:
import sys
import os 

project_root = "/Users/janikwahrheit/Library/CloudStorage/OneDrive-Persönlich/01_Studium/01_Bachelor/Bachelorarbeit/Code"

sys.path.append(project_root)

<h1> Training Pipeline </h1>

In [2]:
from utils.analytics import eval_fit_methods
import networks
import estimator.stable_estimators as se
from scipy.stats import levy_stable
import matplotlib.pyplot as plt 
import seaborn as sns 
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import pandas as pd
import pickle
import numpy as np
import torch
import torch.nn as nn
import utils.optimreg as optimreg
from data.dataloaders import get_mnist_loaders, get_cifar_loaders

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

<h2> Configs </h2>

In [4]:
RUNS = 10
REGULARIZERS = [None, "lasso", "hill", "parabolic_hill", "xiao", "decay"]
LAMBDA_SEARCH = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]

ARCHITECTURES = {
    "fc3_mnist": [784, 256, 256, 256, 10],
    "fc3_cifar": [32*32*3, 256, 256, 256, 10],
    "fc10_mnist": [784] + [256]*10 + [10],
    "fc10_cifar": [32*32*3] + [256]*10 + [10]
}

EPOCHS = {
    "fc3_mnist": 10,
    "fc10_mnist": 200,
    "fc3_cifar": 50,
    "fc10_cifar": 200
}

TUNING_EPOCHS = {
    "fc3_mnist": 5,   
    "fc10_mnist": 20,  
    "fc3_cifar": 10,
    "fc10_cifar": 20
}


<h2> Hyperparameter Tuning </h2> 

In [5]:
def tune_lambda(dataset_name, arch_key, optimizer_name, train_loader, val_loader, reg):
    """
    Führe Hyperparameter-Tuning für lambda durch (nur bei Regularizer != None)
    """
    architecture = ARCHITECTURES[arch_key]

    if reg is None:
        return 0.0  # Kein Tuning für None

    best_acc = -1.0
    best_lambda = LAMBDA_SEARCH[0]

    for lam in LAMBDA_SEARCH:
        model = networks.FCNet(
            layer_sizes=architecture,
            activation='relu',
            weight_init='gaussian'
        ).to(device)

        optimizer = optimreg.get_optimizer(model, optimizer_name=optimizer_name, lr=1e-3)
        epochs = TUNING_EPOCHS[arch_key]

        networks.train(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            epochs=epochs,
            model_name=f"tuning_{arch_key}_{reg}_{lam}_{optimizer_name}_{dataset_name}",
            logging=False,
            run=0,  # nur 1 Run
            regularizer=reg,
            lambda_reg=lam,
            architecture=architecture
        )

        acc = networks.evaluate(model, val_loader)
        if acc > best_acc:
            best_acc = acc
            best_lambda = lam

    print(f"Best lambda for {arch_key} | {reg} | {optimizer_name}: {best_lambda}")
    return best_lambda

<h2> Training Runs </h2>

In [6]:
def run_experiment(dataset_name, arch_key, optimizer_name, train_loader, val_loader, test_loader):
    architecture = ARCHITECTURES[arch_key]

    for reg in REGULARIZERS:
        # Hyperparameter-Tuning direkt vor jedem Regularizer
        lam = tune_lambda(dataset_name, arch_key, optimizer_name, train_loader, val_loader, reg)

        accuracies = []

        for run in range(RUNS):
            print("\n==============================================")
            print(f"Dataset: {dataset_name}")
            print(f"Model: {arch_key}")
            print(f"Regularizer: {reg}")
            print(f"Lambda: {lam}")
            print(f"Optimizer: {optimizer_name}")
            print(f"Run: {run}")
            print("==============================================\n")

            model = networks.FCNet(
                layer_sizes=architecture,
                activation='relu',
                weight_init='gaussian'
            ).to(device)

            optimizer = optimreg.get_optimizer(model, optimizer_name=optimizer_name, lr=1e-3)
            epochs = EPOCHS[arch_key]
            name = f"{arch_key}_{reg}_{optimizer_name}_{dataset_name}"

            networks.train(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                optimizer=optimizer,
                epochs=epochs,
                model_name=name,
                logging=True,
                run=run,
                regularizer=reg,
                lambda_reg=lam,
                architecture=architecture
            )

            acc = networks.evaluate(model, test_loader)
            accuracies.append(acc)

        print("\n================================")
        print(f"Finished: {arch_key} | Reg: {reg} | λ={lam} | Optimizer: {optimizer_name}")
        print("Accuracies:", accuracies)
        print("Mean:", np.mean(accuracies))
        print("Std:", np.std(accuracies))
        print("================================\n")




In [7]:
OPTIMIZERS = ["sgd", "adam"]
BATCH_SIZE = 32


In [ ]:


# MNIST
train_loader, val_loader, test_loader = get_mnist_loaders(batch_size=BATCH_SIZE)
for opt in OPTIMIZERS:
    print(f"\n===== Running MNIST FC3 | Optimizer: {opt} =====")
    run_experiment("mnist", "fc3_mnist", opt, train_loader, val_loader, test_loader)

    print(f"\n===== Running MNIST FC10 | Optimizer: {opt} =====")
    run_experiment("mnist", "fc10_mnist", opt, train_loader, val_loader, test_loader)


100%|██████████| 9.91M/9.91M [00:01<00:00, 6.21MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 241kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.36MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 1.54MB/s]



===== Running MNIST FC3 | Optimizer: sgd =====

Dataset: mnist
Model: fc3_mnist
Regularizer: None
Lambda: 0.0
Optimizer: sgd
Run: 0

Epoch 1/10 | Train Loss: 2.1763 | Train Acc: 33.31% | Val Loss: 1.9939 | Val Acc: 55.21%
Epoch 2/10 | Train Loss: 1.6507 | Train Acc: 64.72% | Val Loss: 1.2609 | Val Acc: 73.07%
Epoch 3/10 | Train Loss: 0.9717 | Train Acc: 78.00% | Val Loss: 0.7643 | Val Acc: 81.44%
Epoch 4/10 | Train Loss: 0.6633 | Train Acc: 83.20% | Val Loss: 0.5808 | Val Acc: 84.58%
Epoch 5/10 | Train Loss: 0.5343 | Train Acc: 85.85% | Val Loss: 0.4901 | Val Acc: 86.48%
Epoch 6/10 | Train Loss: 0.4637 | Train Acc: 87.52% | Val Loss: 0.4359 | Val Acc: 88.02%
Epoch 7/10 | Train Loss: 0.4192 | Train Acc: 88.58% | Val Loss: 0.4004 | Val Acc: 88.78%
Epoch 8/10 | Train Loss: 0.3881 | Train Acc: 89.28% | Val Loss: 0.3738 | Val Acc: 89.66%
Epoch 9/10 | Train Loss: 0.3649 | Train Acc: 89.85% | Val Loss: 0.3554 | Val Acc: 90.08%
Epoch 10/10 | Train Loss: 0.3469 | Train Acc: 90.27% | Val Loss: 

R callback write-console: Fehler in if (is.na(U) | is.na(V)) { : Argument hat Länge 0
  


Epoch 1/10 | Train Loss: 29.9472 | Train Acc: 67.36% | Val Loss: 0.3960 | Val Acc: 87.89%
Epoch 2/10 | Train Loss: 29.2961 | Train Acc: 87.43% | Val Loss: 0.5068 | Val Acc: 86.62%
Epoch 3/10 | Train Loss: 44.6950 | Train Acc: 86.03% | Val Loss: 0.7408 | Val Acc: 86.12%
Epoch 4/10 | Train Loss: 54.2134 | Train Acc: 85.91% | Val Loss: 1.0714 | Val Acc: 87.06%
Epoch 5/10 | Train Loss: 59.2710 | Train Acc: 86.51% | Val Loss: 1.2995 | Val Acc: 87.00%
Epoch 6/10 | Train Loss: 13.6593 | Train Acc: 89.28% | Val Loss: 0.9456 | Val Acc: 89.49%
Epoch 7/10 | Train Loss: 12.6895 | Train Acc: 90.97% | Val Loss: 0.8241 | Val Acc: 90.92%
Epoch 8/10 | Train Loss: 14.5735 | Train Acc: 91.15% | Val Loss: 0.9192 | Val Acc: 89.73%
Epoch 9/10 | Train Loss: 16.8232 | Train Acc: 90.72% | Val Loss: 0.9197 | Val Acc: 90.00%
Epoch 10/10 | Train Loss: 6.6923 | Train Acc: 91.32% | Val Loss: 0.7692 | Val Acc: 91.51%
Test Accuracy: 91.50%

Dataset: mnist
Model: fc3_mnist
Regularizer: hill
Lambda: 0.0003
Optimizer: s

R callback write-console: Fehler in if (is.na(U) | is.na(V)) { : Argument hat Länge 0
  
R callback write-console: Fehler in if (is.na(U) | is.na(V)) { : Argument hat Länge 0
  


...

Epoch 96/200 | Train Loss: 4.3334 | Train Acc: 98.89% | Val Loss: 0.4167 | Val Acc: 94.71%


R callback write-console: Fehler in if (is.na(U) | is.na(V)) { : Argument hat Länge 0
  


Epoch 97/200 | Train Loss: 4.3456 | Train Acc: 98.66% | Val Loss: 0.3424 | Val Acc: 95.10%


R callback write-console: Fehler in if (is.na(U) | is.na(V)) { : Argument hat Länge 0
  


Epoch 98/200 | Train Loss: nan | Train Acc: 31.16% | Val Loss: nan | Val Acc: 9.71%
Epoch 99/200 | Train Loss: nan | Train Acc: 9.91% | Val Loss: nan | Val Acc: 9.71%
Epoch 100/200 | Train Loss: nan | Train Acc: 9.91% | Val Loss: nan | Val Acc: 9.71%
Epoch 101/200 | Train Loss: nan | Train Acc: 9.91% | Val Loss: nan | Val Acc: 9.71%
Epoch 102/200 | Train Loss: nan | Train Acc: 9.91% | Val Loss: nan | Val Acc: 9.71%
Epoch 103/200 | Train Loss: nan | Train Acc: 9.91% | Val Loss: nan | Val Acc: 9.71%
Epoch 104/200 | Train Loss: nan | Train Acc: 9.91% | Val Loss: nan | Val Acc: 9.71%
Epoch 105/200 | Train Loss: nan | Train Acc: 9.91% | Val Loss: nan | Val Acc: 9.71%
Epoch 106/200 | Train Loss: nan | Train Acc: 9.91% | Val Loss: nan | Val Acc: 9.71%
Epoch 107/200 | Train Loss: nan | Train Acc: 9.91% | Val Loss: nan | Val Acc: 9.71%
Epoch 108/200 | Train Loss: nan | Train Acc: 9.91% | Val Loss: nan | Val Acc: 9.71%
Epoch 109/200 | Train Loss: nan | Train Acc: 9.91% | Val Loss: nan | Val Acc:

In [ ]:

# CIFAR
train_loader, val_loader, test_loader = get_cifar_loaders(batch_size=BATCH_SIZE)
for opt in OPTIMIZERS:
    print(f"\n===== Running CIFAR FC3 | Optimizer: {opt} =====")
    run_experiment("cifar", "fc3_cifar", opt, train_loader, val_loader, test_loader)

    print(f"\n===== Running CIFAR FC10 | Optimizer: {opt} =====")
    run_experiment("cifar", "fc10_cifar", opt, train_loader, val_loader, test_loader)

100%|██████████| 170M/170M [00:16<00:00, 10.4MB/s] 



===== Running CIFAR FC3 | Optimizer: sgd =====

Dataset: cifar
Model: fc3_cifar
Regularizer: None
Lambda: 0.0
Optimizer: sgd
Run: 0

Epoch 1/50 | Train Loss: 2.2179 | Train Acc: 18.91% | Val Loss: 2.1420 | Val Acc: 22.08%
Epoch 2/50 | Train Loss: 2.0693 | Train Acc: 26.73% | Val Loss: 2.0180 | Val Acc: 28.15%
Epoch 3/50 | Train Loss: 1.9716 | Train Acc: 30.12% | Val Loss: 1.9473 | Val Acc: 31.29%
Epoch 4/50 | Train Loss: 1.9117 | Train Acc: 32.26% | Val Loss: 1.8985 | Val Acc: 32.07%
Epoch 5/50 | Train Loss: 1.8678 | Train Acc: 34.29% | Val Loss: 1.8570 | Val Acc: 34.16%
Epoch 6/50 | Train Loss: 1.8334 | Train Acc: 35.40% | Val Loss: 1.8270 | Val Acc: 35.25%
Epoch 7/50 | Train Loss: 1.8045 | Train Acc: 36.47% | Val Loss: 1.7984 | Val Acc: 35.89%
Epoch 8/50 | Train Loss: 1.7778 | Train Acc: 37.26% | Val Loss: 1.7773 | Val Acc: 37.39%
Epoch 9/50 | Train Loss: 1.7553 | Train Acc: 38.32% | Val Loss: 1.7551 | Val Acc: 37.52%
Epoch 10/50 | Train Loss: 1.7344 | Train Acc: 39.07% | Val Loss: 

R callback write-console: Fehler in if (is.na(U) | is.na(V)) { : Argument hat Länge 0
  


Epoch 17/50 | Train Loss: 1.6294 | Train Acc: 42.72% | Val Loss: 1.6392 | Val Acc: 42.07%
Epoch 18/50 | Train Loss: 1.6165 | Train Acc: 43.14% | Val Loss: 1.6320 | Val Acc: 42.56%
Epoch 19/50 | Train Loss: 1.6041 | Train Acc: 43.54% | Val Loss: 1.6213 | Val Acc: 42.49%
Epoch 20/50 | Train Loss: 1.5920 | Train Acc: 43.99% | Val Loss: 1.6142 | Val Acc: 42.78%
Epoch 21/50 | Train Loss: 1.5808 | Train Acc: 44.48% | Val Loss: 1.6000 | Val Acc: 43.38%
Epoch 22/50 | Train Loss: 1.5704 | Train Acc: 44.77% | Val Loss: 1.5908 | Val Acc: 43.84%
Epoch 23/50 | Train Loss: 1.5599 | Train Acc: 45.05% | Val Loss: 1.5851 | Val Acc: 44.43%
Epoch 24/50 | Train Loss: 1.5497 | Train Acc: 45.45% | Val Loss: 1.5741 | Val Acc: 44.27%
Epoch 25/50 | Train Loss: 1.5401 | Train Acc: 45.78% | Val Loss: 1.5678 | Val Acc: 44.76%
Epoch 26/50 | Train Loss: 1.5300 | Train Acc: 46.22% | Val Loss: 1.5658 | Val Acc: 44.44%
Epoch 27/50 | Train Loss: 1.5216 | Train Acc: 46.45% | Val Loss: 1.5522 | Val Acc: 45.28%
Epoch 28/5

R callback write-console: Fehler in if (is.na(U) | is.na(V)) { : Argument hat Länge 0
  


...

Epoch 2/50 | Train Loss: 28.8709 | Train Acc: 30.08% | Val Loss: 1.9828 | Val Acc: 29.79%
Epoch 3/50 | Train Loss: 39.2922 | Train Acc: 28.27% | Val Loss: 2.2043 | Val Acc: 24.02%
Epoch 4/50 | Train Loss: 53.0597 | Train Acc: 26.67% | Val Loss: 2.2444 | Val Acc: 23.34%
Epoch 5/50 | Train Loss: 11.5287 | Train Acc: 27.81% | Val Loss: 1.9387 | Val Acc: 30.50%
Epoch 6/50 | Train Loss: 8.3464 | Train Acc: 32.12% | Val Loss: 1.8815 | Val Acc: 32.51%
Epoch 7/50 | Train Loss: 8.1408 | Train Acc: 34.23% | Val Loss: 1.8611 | Val Acc: 33.62%
Epoch 8/50 | Train Loss: 8.0097 | Train Acc: 35.34% | Val Loss: 1.8291 | Val Acc: 35.51%
Epoch 9/50 | Train Loss: 7.9429 | Train Acc: 36.13% | Val Loss: 1.8100 | Val Acc: 36.12%
Epoch 10/50 | Train Loss: 7.9219 | Train Acc: 36.61% | Val Loss: 1.8072 | Val Acc: 36.38%
Epoch 11/50 | Train Loss: 7.9376 | Train Acc: 37.27% | Val Loss: 1.7812 | Val Acc: 37.10%
Epoch 12/50 | Train Loss: 7.9857 | Train Acc: 38.00% | Val Loss: 1.7804 | Val Acc: 36.68%
Epoch 13/50 | 

R callback write-console: Fehler in if (is.na(U) | is.na(V)) { : Argument hat Länge 0
  


Epoch 28/50 | Train Loss: 1.5985 | Train Acc: 42.98% | Val Loss: 1.6799 | Val Acc: 40.56%
Epoch 29/50 | Train Loss: 1.5935 | Train Acc: 42.91% | Val Loss: 1.6712 | Val Acc: 41.07%
Epoch 30/50 | Train Loss: 1.5925 | Train Acc: 42.90% | Val Loss: 1.6475 | Val Acc: 42.13%
Epoch 31/50 | Train Loss: 1.5884 | Train Acc: 43.04% | Val Loss: 1.6782 | Val Acc: 41.58%
Epoch 32/50 | Train Loss: 1.5839 | Train Acc: 43.65% | Val Loss: 1.6537 | Val Acc: 41.53%
Epoch 33/50 | Train Loss: 1.5811 | Train Acc: 43.58% | Val Loss: 1.6537 | Val Acc: 41.98%
Epoch 34/50 | Train Loss: 1.5762 | Train Acc: 43.59% | Val Loss: 1.6557 | Val Acc: 41.81%
Epoch 35/50 | Train Loss: 1.5724 | Train Acc: 43.85% | Val Loss: 1.6652 | Val Acc: 41.15%
Epoch 36/50 | Train Loss: 1.5685 | Train Acc: 43.81% | Val Loss: 1.6489 | Val Acc: 41.88%
Epoch 37/50 | Train Loss: 1.5643 | Train Acc: 44.07% | Val Loss: 1.6699 | Val Acc: 41.16%
Epoch 38/50 | Train Loss: 1.5625 | Train Acc: 44.40% | Val Loss: 1.6500 | Val Acc: 41.60%
Epoch 39/5

R callback write-console: Fehler in if (is.na(U) | is.na(V)) { : Argument hat Länge 0
  


Epoch 7/50 | Train Loss: 1.8599 | Train Acc: 32.73% | Val Loss: 1.8766 | Val Acc: 32.15%
Epoch 8/50 | Train Loss: 1.8536 | Train Acc: 32.41% | Val Loss: 1.9158 | Val Acc: 30.02%
Epoch 9/50 | Train Loss: 1.8436 | Train Acc: 32.95% | Val Loss: 1.9408 | Val Acc: 29.91%
Epoch 10/50 | Train Loss: 1.8362 | Train Acc: 33.36% | Val Loss: 1.8946 | Val Acc: 32.18%
Epoch 11/50 | Train Loss: 1.8274 | Train Acc: 33.85% | Val Loss: 1.8480 | Val Acc: 33.26%
Epoch 12/50 | Train Loss: 1.8231 | Train Acc: 33.87% | Val Loss: 1.8701 | Val Acc: 32.35%
Epoch 13/50 | Train Loss: 1.8144 | Train Acc: 34.02% | Val Loss: 1.8333 | Val Acc: 34.24%
Epoch 14/50 | Train Loss: 1.8109 | Train Acc: 34.29% | Val Loss: 1.8553 | Val Acc: 33.10%
Epoch 15/50 | Train Loss: 1.8056 | Train Acc: 34.67% | Val Loss: 1.8481 | Val Acc: 33.83%
Epoch 16/50 | Train Loss: 1.7947 | Train Acc: 34.92% | Val Loss: 1.8458 | Val Acc: 32.88%
Epoch 17/50 | Train Loss: 1.7930 | Train Acc: 34.84% | Val Loss: 1.8136 | Val Acc: 34.67%
Epoch 18/50 |